## Figure 4 — MERRA-2 2020 and WACCM year-0008 NAM/O3 context

Inputs: canonical cases/figure04_context.nc. The compact diagnostic
product stores the fixed-EOF vertical NAM, its 1000-hPa AO slice,
30–70-hPa/60–90°N partial O3, target-excluded calendar-day anomalies,
the centered 5-day anomaly, profile anomalies, and the stored
March–April minimum for both events. MERRA-2 is left and WACCM is
right; no climatology, smoothing, minimum, NAM, or AO is recalculated.

Outputs:
figure04_merra2_2020_waccm0008_nam_o3_context_MERRA2NEWNAM.png and PDF.

Plot action: validate the compact product and display its stored NAM,
O3-profile contours, AO series, partial-O3 anomaly, and minimum marker.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

def discover_repository_root() -> Path:
    """Locate the cloned ``code`` directory from a notebook kernel."""

    candidates = (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent)
    for candidate in candidates:
        if (candidate / "02_diagnostics").is_dir() and (candidate / "03_plotting").is_dir():
            return candidate.resolve()
    raise RuntimeError(
        "Cannot locate the Paper 1 code directory. Run the notebook from the "
        "code directory or one of its notebook subdirectories."
    )


REPOSITORY_ROOT = discover_repository_root()
DEFAULT_DERIVED_ROOT = REPOSITORY_ROOT / "runtime"
REPOSITORY_RUNTIME_ROOT = DEFAULT_DERIVED_ROOT.resolve()
PUBLIC_ROOT = Path("/mnt/soclim0/public_data/weiji").resolve()
PROTECTED_DESTINATIONS = tuple(
    path.resolve()
    for path in (
        Path("/mnt/backup_ETH"),
        PUBLIC_ROOT / "B2000WCN001002_timefixed",
        PUBLIC_ROOT / "BWCN",
        PUBLIC_ROOT / "Hindcast",
        PUBLIC_ROOT / "Marina",
        PUBLIC_ROOT / "MERRA2M2I6NPANA",
        PUBLIC_ROOT / "MERRA2_Processed",
        PUBLIC_ROOT / "MLS",
        PUBLIC_ROOT / "CO2x1SmidEmin_yBWCN_timefixed",
    )
)
PRODUCT_VERSION = "Paper1_828_repro_v1"


def is_within(candidate: Path, parent: Path) -> bool:
    return candidate == parent or parent in candidate.parents


def validate_staging_root(candidate: Path) -> Path:
    root = candidate.expanduser().resolve()
    if root == Path(root.anchor) or root == PUBLIC_ROOT:
        raise PermissionError(f"Refusing unsafe staging root: {root}")
    if REPOSITORY_ROOT.is_dir() and not is_within(root, REPOSITORY_RUNTIME_ROOT):
        raise PermissionError(
            "PAPER1_DERIVED_ROOT must remain below this checkout's dedicated "
            f"runtime tree: root={root}, scope={REPOSITORY_RUNTIME_ROOT}"
        )
    for protected in PROTECTED_DESTINATIONS:
        if is_within(root, protected) or is_within(protected, root):
            raise PermissionError(
                "Refusing staging root that overlaps protected raw/legacy "
                f"tree: root={root}, protected={protected}"
            )
    return root


def canonical_root() -> Path:
    # Return the dedicated, ignored runtime tree (or an explicit safe child).
    override = os.environ.get("PAPER1_DERIVED_ROOT")
    if override:
        return validate_staging_root(Path(override))
    return validate_staging_root(DEFAULT_DERIVED_ROOT)


DERIVED_ROOT = canonical_root()
OUTPUT_DIR = Path(
    os.environ.get("PAPER1_FIGURE_ROOT", str(DERIVED_ROOT / "figures"))
).expanduser().resolve()
if not is_within(OUTPUT_DIR, DERIVED_ROOT):
    raise PermissionError(
        f"PAPER1_FIGURE_ROOT must remain below PAPER1_DERIVED_ROOT: {OUTPUT_DIR}"
    )


def canonical_path(relative: str) -> Path:
    path = DERIVED_ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing canonical Paper 1 product: {path}. "
            "Run 02_diagnostics on STREAM2 or set PAPER1_DERIVED_ROOT."
        )
    return path


def load_dataset(
    relative: str,
    required_variables: tuple[str, ...],
    *,
    require_version: bool = True,
) -> xr.Dataset:
    path = canonical_path(relative)
    with xr.open_dataset(path, decode_times=False) as opened:
        dataset = opened.load()
    missing = [name for name in required_variables if name not in dataset]
    if missing:
        raise ValueError(f"{path} is missing canonical variables {missing}")
    if require_version and dataset.attrs.get("product_version") != PRODUCT_VERSION:
        raise ValueError(
            f"{path}: product_version={dataset.attrs.get('product_version')!r}; "
            f"expected {PRODUCT_VERSION!r}"
        )
    return dataset


def require_columns(
    frame: pd.DataFrame, path: Path, columns: tuple[str, ...]
) -> None:
    missing = [name for name in columns if name not in frame]
    if missing:
        raise ValueError(f"{path} is missing canonical columns {missing}")
    if "product_version" not in frame:
        raise ValueError(f"{path} is missing product_version")
    versions = set(frame["product_version"].dropna().astype(str))
    if versions != {PRODUCT_VERSION}:
        raise ValueError(
            f"{path}: product_version values {sorted(versions)!r}; "
            f"expected only {PRODUCT_VERSION!r}"
        )


def canonical_waccm_threshold() -> float:
    # Read, but never reconstruct, the fixed low-25 threshold from 230 springs.
    path = canonical_path("ozone/waccm_master_rankings.csv")
    ranking = pd.read_csv(path)
    require_columns(
        ranking, path,
        (
            "sample_size", "low25_count", "low25_threshold_du",
            "is_low25",
        ),
    )
    if len(ranking) != 230:
        raise ValueError(f"{path}: expected exactly 230 ranked springs")
    if set(ranking["sample_size"].astype(int)) != {230}:
        raise ValueError(f"{path}: sample_size must be 230 on every row")
    if set(ranking["low25_count"].astype(int)) != {57}:
        raise ValueError(f"{path}: fixed low25 count must be floor(230/4)=57")
    thresholds = pd.to_numeric(
        ranking["low25_threshold_du"], errors="raise"
    ).unique()
    if len(thresholds) != 1 or not np.isfinite(thresholds[0]):
        raise ValueError(f"{path}: expected one finite fixed threshold")
    return float(thresholds[0])


def validate_fixed_threshold(values: pd.Series, path: Path) -> float:
    stored = pd.to_numeric(values, errors="raise").unique()
    if len(stored) != 1 or not np.isfinite(stored[0]):
        raise ValueError(f"{path}: expected one finite stored low25 threshold")
    master = canonical_waccm_threshold()
    if not np.isclose(float(stored[0]), master, rtol=0.0, atol=1e-10):
        raise ValueError(
            f"{path}: threshold {stored[0]} differs from 230-spring "
            f"master threshold {master}"
        )
    return master


def parse_boolean(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values
    if pd.api.types.is_numeric_dtype(values):
        return values.astype(int).astype(bool)
    mapping = {"true": True, "false": False, "1": True, "0": False}
    parsed = values.astype(str).str.strip().str.lower().map(mapping)
    if parsed.isna().any():
        raise ValueError(
            f"Cannot parse boolean values {values[parsed.isna()].unique()}"
        )
    return parsed.astype(bool)


def text_value(value: object) -> str:
    # Decode NetCDF byte-string coordinates without producing "b'...'" labels.
    if isinstance(value, (bytes, np.bytes_)):
        return value.decode("utf-8")
    return str(value)


def date_parts(
    date_variable: xr.DataArray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    values = np.asarray(date_variable.values)
    if values.ndim != 1:
        raise ValueError(f"date must be one-dimensional, found {values.shape}")
    if np.issubdtype(values.dtype, np.datetime64):
        dates = pd.DatetimeIndex(values)
        return (
            dates.year.to_numpy(dtype=int),
            dates.month.to_numpy(dtype=int),
            dates.day.to_numpy(dtype=int),
        )
    compact = values.astype(np.int64)
    return compact // 10000, (compact % 10000) // 100, compact % 100


def pressure_slice(
    values: xr.DataArray, pressure_hpa: float
) -> xr.DataArray:
    if "plev" not in values.dims:
        raise ValueError(
            f"Expected canonical plev dimension, found {values.dims}"
        )
    levels = np.asarray(values["plev"].values, dtype=float)
    if np.nanmax(levels) > 1100.0:
        raise ValueError("Canonical plev must be stored in hPa")
    selected = values.sel(plev=float(pressure_hpa), method="nearest")
    actual = float(selected["plev"])
    tolerance = max(0.6, pressure_hpa * 0.02)
    if not np.isclose(actual, pressure_hpa, rtol=0.0, atol=tolerance):
        raise ValueError(
            f"Requested {pressure_hpa} hPa; nearest level is {actual}"
        )
    return selected


def save_figure(
    figure: plt.Figure, stem: str, *, dpi: int = 300
) -> None:
    if Path(stem).name != stem or not stem:
        raise ValueError(f"Figure stem must be one safe filename: {stem!r}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    png = OUTPUT_DIR / f"{stem}.png"
    pdf = OUTPUT_DIR / f"{stem}.pdf"
    temporary_png = OUTPUT_DIR / f".{stem}.{os.getpid()}.png.tmp"
    temporary_pdf = OUTPUT_DIR / f".{stem}.{os.getpid()}.pdf.tmp"
    try:
        figure.savefig(
            temporary_png, format="png", dpi=dpi,
            bbox_inches="tight", facecolor="white",
        )
        figure.savefig(
            temporary_pdf, format="pdf", bbox_inches="tight",
            facecolor="white",
        )
        if temporary_png.stat().st_size < 1024 or temporary_pdf.stat().st_size < 1024:
            raise RuntimeError(f"Rendered output is unexpectedly small: {stem}")
        os.replace(temporary_png, png)
        os.replace(temporary_pdf, pdf)
    finally:
        plt.close(figure)
        for temporary in (temporary_png, temporary_pdf):
            if temporary.exists():
                temporary.unlink()
    print(f"saved {png}")
    print(f"saved {pdf}")
product = load_dataset(
    "cases/figure04_context.nc",
    (
        "nam", "ao", "partial_o3_du",
        "partial_o3_anomaly_rm5_du", "o3_profile_anomaly",
        "minimum_du", "minimum_anomaly_du", "minimum_event_day",
    ),
)
expected_dims = {
    "nam": ("source", "event_day", "plev"),
    "ao": ("source", "event_day"),
    "partial_o3_du": ("source", "event_day"),
    "partial_o3_anomaly_rm5_du": ("source", "event_day"),
    "o3_profile_anomaly": (
        "source", "event_day", "profile_pressure_hpa"
    ),
    "minimum_du": ("source",),
    "minimum_anomaly_du": ("source",),
    "minimum_event_day": ("source",),
}
for variable, dimensions in expected_dims.items():
    if product[variable].dims != dimensions:
        raise ValueError(
            f"cases/figure04_context.nc: {variable} dimensions "
            f"{product[variable].dims}; expected {dimensions}"
        )

event_day = np.asarray(product["event_day"].values, dtype=int)
if not np.array_equal(event_day, np.arange(212)):
    raise ValueError("Figure 4 event_day must be Nov 1–May 31 (0..211)")
pressure = np.asarray(product["plev"].values, dtype=float)
profile_pressure = np.asarray(
    product["profile_pressure_hpa"].values, dtype=float
)
if np.nanmax(pressure) > 1100.0 or np.nanmin(pressure) <= 0:
    raise ValueError("Figure 4 NAM pressure must be positive hPa")
if (
    np.nanmin(profile_pressure) < 0.9
    or np.nanmax(profile_pressure) > 101.0
):
    raise ValueError("Figure 4 O3-profile pressure must span 1–100 hPa")

metadata = " ".join(
    f"{key}={value}" for key, value in product.attrs.items()
).lower()
method_tokens = (
    "target", "excluded", "centered", "30", "70",
    "60", "90", "1000",
)
missing_tokens = [token for token in method_tokens if token not in metadata]
if missing_tokens:
    raise ValueError(
        "Figure 4 metadata does not document all canonical methods; "
        f"missing tokens {missing_tokens}"
    )

source_labels = [text_value(value) for value in product["source"].values]
observed = [
    label for label in source_labels if "merra" in label.lower()
]
modeled = [
    label for label in source_labels if "waccm" in label.lower()
]
if len(observed) != 1 or len(modeled) != 1:
    raise ValueError(f"Unexpected Figure 4 sources {source_labels}")
ordered_sources = observed + modeled

figure, axes = plt.subplots(
    3, 2, figsize=(15.2, 10.1),
    gridspec_kw={"height_ratios": [2.25, 0.70, 0.90]},
    constrained_layout=True,
)
mappable = None
for column, source in enumerate(ordered_sources):
    nam_values = np.asarray(
        product["nam"].sel(source=source).values, dtype=float
    )
    profile_values = np.asarray(
        product["o3_profile_anomaly"].sel(source=source).values,
        dtype=float,
    )
    mappable = axes[0, column].contourf(
        event_day, pressure, nam_values.T,
        levels=np.arange(-4.5, 5.0, 0.5),
        cmap="RdBu_r", extend="both",
    )
    profile_contour = axes[0, column].contour(
        event_day, profile_pressure, profile_values.T,
        levels=[-1.0, -0.5], colors="#ff2d95",
        linewidths=1.05, linestyles="--",
    )
    axes[0, column].clabel(
        profile_contour, fontsize=6.5, fmt="%g"
    )
    axes[0, column].set_yscale("log")
    axes[0, column].invert_yaxis()
    axes[0, column].set_ylim(1000, 1)
    axes[0, column].set_title(
        "MERRA-2 2020" if column == 0 else "WACCM year 0008",
        loc="left", fontweight="bold",
    )

    ao = np.asarray(
        product["ao"].sel(source=source).values, dtype=float
    )
    axes[1, column].plot(event_day, ao, color="0.10", lw=1.6)
    axes[1, column].axhline(0, color="0.55", lw=0.7)
    axes[1, column].set_ylabel("AO = NAM at 1000 hPa")

    anomaly = np.asarray(
        product["partial_o3_anomaly_rm5_du"]
        .sel(source=source).values,
        dtype=float,
    )
    minimum_day = int(
        product["minimum_event_day"].sel(source=source).item()
    )
    minimum_anomaly = float(
        product["minimum_anomaly_du"].sel(source=source).item()
    )
    minimum_du = float(
        product["minimum_du"].sel(source=source).item()
    )
    if minimum_day < 120 or minimum_day > 180:
        raise ValueError(
            f"{source}: stored O3 minimum is outside March–April"
        )
    axes[2, column].plot(
        event_day, anomaly, color="#7b2cbf", lw=1.8
    )
    axes[2, column].scatter(
        minimum_day, minimum_anomaly,
        s=42, color="#d62728", zorder=5,
    )
    axes[2, column].axvline(
        minimum_day, color="#d62728", ls="--", lw=1.0
    )
    axes[2, column].axhline(0, color="0.55", lw=0.7)
    axes[2, column].text(
        0.98, 0.06, f"stored minimum = {minimum_du:.1f} DU",
        transform=axes[2, column].transAxes,
        ha="right", va="bottom", fontsize=8,
    )
    axes[2, column].set_ylabel(
        "Centered 5-day partial O$_3$\nanomaly (DU)"
    )

    for row in range(3):
        axes[row, column].set_xlim(0, 211)
        axes[row, column].set_xticks(
            [0, 30, 61, 92, 120, 151, 181],
            [] if row < 2 else
            ["Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May"],
        )
        axes[row, column].grid(
            axis="x", color="0.88", lw=0.45
        )
axes[0, 0].set_ylabel("Pressure (hPa)")
colorbar = figure.colorbar(mappable, ax=axes[0, :], pad=0.02)
colorbar.set_label("NAM index")
figure.suptitle(
    "MERRA-2 2020 and WACCM year-0008 NAM/AO/O$_3$ context",
    fontsize=14.5, fontweight="bold",
)
save_figure(
    figure,
    "figure04_merra2_2020_waccm0008_nam_o3_context_MERRA2NEWNAM",
)
